In [ ]:
# @title 1. 克隆仓库 & 安装依赖
!git clone -q https://github.com/TheOne1006/timm-cchess-reg.git || true
%cd timm-cchess-reg

!pip install -q timm accelerate

In [ ]:
# @title 2. 挂载 Google Drive & 准备数据
from google.colab import drive
drive.mount('/content/drive')

import os

# 解压数据集（首次运行后会缓存，后续跳过）
if not os.path.exists("datasets/full"):
    !mkdir -p datasets
    !cp "/content/drive/MyDrive/Colab Notebooks/full.zip" datasets/
    !cd datasets && unzip -q full.zip
    print("数据集已解压")
else:
    print("数据集已存在，跳过解压")

# 解压棋子 PNG（用于 PiecePaste 增强，可选）
if not os.path.exists("datasets/single_cls2_png"):
    !cp "/content/drive/MyDrive/Colab Notebooks/single_cls2_png.zip" datasets/ 2>/dev/null || true
    !cd datasets && unzip -q single_cls2_png.zip 2>/dev/null || true

DATA_DIR = "datasets/full"
PNG_DIR = "datasets/single_cls2_png" if os.path.exists("datasets/single_cls2_png") else None
print(f"数据目录: {DATA_DIR}, PNG目录: {PNG_DIR}")

In [ ]:
# @title 3. 数据增强预览
import sys
import random
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from src.dataset import CChessDataset, IDX_TO_FEN_CHAR, BOARD_ROWS, BOARD_COLS
from src.transforms.pipeline import train_transform
from src.transforms.base import ToTensorNormalize
from src.visualize_transforms import capture_stages, draw_board_overlay

# 构建 transform pipeline
transform = train_transform(png_dir=PNG_DIR)

# 随机选一张图
dataset = CChessDataset(root=DATA_DIR, transform=None)
idx = random.randint(0, len(dataset) - 1)
image, label = dataset[idx]
print(f"样本 #{idx}, 图像尺寸: {image.size}, 标签形状: {label.shape}")

# 捕获各阶段
stages = capture_stages(transform, image, label)
all_stages = [("0: Original", image, label)] + stages

# 可视化（只显示前 6 个阶段，避免太多子图）
show_stages = all_stages[:6]
n = len(show_stages)
cols = min(3, n)
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(7 * cols, 6 * rows))
axes_flat = np.array(axes).flatten() if n > 1 else [axes]

for i, (name, img, lbl) in enumerate(show_stages):
    overlay = draw_board_overlay(img, lbl)
    axes_flat[i].imshow(np.array(overlay))
    axes_flat[i].set_title(name, fontsize=9)
    axes_flat[i].axis("off")

for j in range(n, len(axes_flat)):
    axes_flat[j].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# @title 4. 模型结构
import torch
from src.model import CChessNet

model = CChessNet()
model.eval()

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"总参数量: {total_params:,} ({total_params / 1e6:.2f}M)")
print(f"可训练参数: {trainable_params:,} ({trainable_params / 1e6:.2f}M)")
print(f"输入尺寸: [1, 3, {CChessNet.INPUT_HEIGHT}, {CChessNet.INPUT_WIDTH}]")

# 前向传播验证
dummy = torch.randn(1, 3, CChessNet.INPUT_HEIGHT, CChessNet.INPUT_WIDTH)
with torch.no_grad():
    out = model(dummy)
print(f"输出尺寸: {out.shape}")
print(f"softmax 验证 (sum≈1.0): {out[0, 0, 0].sum().item():.4f}")

# 模型结构摘要
print("\n--- 模型结构 ---")
print(model)

In [ ]:
# @title 5. 训练配置 & 启动
import argparse

# @markdown ### 训练参数
epochs = 100  # @param {type:"integer"}
batch_size = 64  # @param {type:"integer"}
learning_rate = 0.0001  # @param {type:"number"}
freeze_backbone_epochs = 10  # @param {type:"integer"} 冻结 backbone 的 epoch 数 (0=不冻结)
backbone_lr_scale = 0.1  # @param {type:"number"} backbone 学习率缩放因子
fp16 = True  # @param {type:"boolean"}
resume_from = ""  # @param {type:"string"} 留空=从头训练，填 checkpoint 路径=恢复训练

args = argparse.Namespace(
    data_dir=DATA_DIR,
    png_dir=PNG_DIR,
    val_ratio=0.1,
    seed=42,
    backbone="convnext_atto.d2_in1k",
    epochs=epochs,
    batch_size=batch_size,
    lr=learning_rate,
    weight_decay=0.05,
    eps=1e-8,
    warmup_epochs=5,
    freeze_backbone_epochs=freeze_backbone_epochs,
    backbone_lr_scale=backbone_lr_scale,
    scheduler="cosine",
    fp16=fp16,
    num_workers=2,
    perspective_prob=0.7,
    piece_paste_prob=0.7 if PNG_DIR else 0.0,
    piece_max_cells=15,
    resume_from=resume_from if resume_from else None,
    output_dir="/content/drive/MyDrive/cchess_outputs",
    log_interval=10,
    report_to="tensorboard",
)

print(f"训练配置: epochs={epochs}, batch_size={batch_size}, lr={learning_rate}, fp16={fp16}")
print(f"Backbone: {args.backbone}, freeze={freeze_backbone_epochs} epochs, lr_scale={backbone_lr_scale}")
print(f"数据集: {DATA_DIR}, 增强PNG: {'是' if PNG_DIR else '否'}")
print(f"输出目录: {args.output_dir}")
if args.resume_from:
    print(f"恢复训练: {args.resume_from}")
else:
    print("从头开始训练")

In [ ]:
# @title 启动 TensorBoard
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/cchess_outputs/runs

In [ ]:
# @title 开始训练
from src.train import train
train(args)

In [ ]:
# @title 7. 训练结果评估
import torch
import numpy as np
from src.model import CChessNet
from src.train import HFModelWrapper, SubsetWithTransform
from src.dataset import CChessDataset, IDX_TO_FEN_CHAR
from src.transforms import val_transform
from src.evaluate import CChessEvaluator
from torch.utils.data import DataLoader, random_split

OUTPUT_DIR = "/content/drive/MyDrive/cchess_outputs"

# 加载最佳模型
model = CChessNet()
state_dict = torch.load(f"{OUTPUT_DIR}/best_model/pytorch_model.bin", map_location="cpu", weights_only=True)
model.load_state_dict(state_dict)
model.eval()

if torch.cuda.is_available():
    model = model.cuda()
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"模型已加载，设备: {device}")

# 准备验证集
full_dataset = CChessDataset(root=DATA_DIR, transform=None)
n_total = len(full_dataset)
n_val = max(1, int(n_total * 0.1))
n_train = n_total - n_val

generator = torch.Generator().manual_seed(42)
_, val_indices = random_split(range(n_total), [n_train, n_val], generator=generator)
val_dataset = SubsetWithTransform(full_dataset, val_indices, val_transform())
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

# 跑评估
evaluator = CChessEvaluator()
model.eval()
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)
        output = model(images, labels=labels)
        evaluator.add_batch(output["logits"], labels)

metrics = evaluator.compute()

# 打印关键指标
print("\n--- 评估结果 ---")
print(f"全盘准确率 (full_accuracy): {metrics['full_accuracy']:.2f}%")
print(f"容错准确率 (err1): {metrics['err1_accuracy']:.2f}%")
print(f"容错准确率 (err3): {metrics['err3_accuracy']:.2f}%")
print(f"容错准确率 (err5): {metrics['err5_accuracy']:.2f}%")
print(f"mAP: {metrics['mAP']:.2f}%")
print(f"piece_only_mAP: {metrics['piece_only_mAP']:.2f}%")
print(f"position_mAP: {metrics['position_mAP']:.2f}%")
print(f"micro_accuracy: {metrics['micro_accuracy']:.2f}%")
print(f"macro_precision: {metrics['macro_precision']:.2f}%")
print(f"macro_recall: {metrics['macro_recall']:.2f}%")
print(f"macro_f1: {metrics['macro_f1']:.2f}%")

# 打印各类别 AP
print("\n--- 各类别 AP ---")
for i in range(16):
    char = IDX_TO_FEN_CHAR[i]
    ap_key = f"class_AP_{char}"
    if ap_key in metrics:
        print(f"  {char:>2s}: {metrics[ap_key]:.2f}%")

In [ ]:
# @title 7. 推理 Demo
from src.inference import print_board
from src.transforms import val_transform
from src.dataset import parse_fen_label

# 选一张 demo 图做推理
demo_path = None
for f in sorted(os.listdir("datasets/demo")):
    if f.endswith(".jpg"):
        demo_path = os.path.join("datasets/demo", f)
        break

if demo_path:
    print(f"推理图片: {demo_path}")
    img = Image.open(demo_path).convert("RGB")
    label_path = demo_path.replace(".jpg", ".txt")
    label = parse_fen_label(open(label_path).read())

    tfm = val_transform()
    tensor_img, _ = tfm(img, label)
    tensor_img = tensor_img.unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        probs = model(tensor_img)

    print_board(probs.cpu())

    # 可视化对比
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    overlay = draw_board_overlay(img, label)
    ax1.imshow(np.array(overlay))
    ax1.set_title("原始标注")
    ax1.axis("off")

    pred_classes = probs[0].argmax(dim=-1).cpu()
    overlay_pred = draw_board_overlay(img, pred_classes)
    ax2.imshow(np.array(overlay_pred))
    ax2.set_title("模型预测")
    ax2.axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("未找到 demo 图片")